# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> This notebook stitches together what w03, w04, w06, and w07 already proved out independently — nothing here is a new experiment, it's the validated pipeline run start to finish in one place.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EsarFatima/MachineLearning-flyrank-"
REPO_DIR = "MachineLearning-flyrank-"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

print("Working directory:", os.getcwd())

Working directory: /content/MachineLearning-flyrank-


**Question:** among a client's existing content pages, which ones are most likely already declining in search visibility right now — and can a validated model rank that risk more usefully than a simple, transparent staleness-and-visibility rule?

**Decision it supports:** which pages a content/SEO editor reviews for a refresh first, out of a portfolio too large to review by hand every cycle. The output is a ranked review queue with a reason code per page, not an automated rewrite or publish action (see Section 5).

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
import pandas as pd, numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "pages,", df.shape[1], "columns,", df["client_id"].nunique(), "pseudonymized clients")
print("month column, if present:", "month" in df.columns)
print()
print("Columns excluded from modeling and why:")
print(" - trend_direction, trend_pct : these BUILD the label -- confession-tested in w03, would leak")
print(" - impressions_last_30d, impressions_prev_30d : the two raw components trend_pct is computed from")
print(" - client_id : used ONLY to group the train/test split, never as a feature")
print(" - content_id : identifier only, not predictive")

30000 pages, 45 columns, 32 pseudonymized clients
month column, if present: False

Columns excluded from modeling and why:
 - trend_direction, trend_pct : these BUILD the label -- confession-tested in w03, would leak
 - impressions_last_30d, impressions_prev_30d : the two raw components trend_pct is computed from
 - client_id : used ONLY to group the train/test split, never as a feature
 - content_id : identifier only, not predictive


**Release:** the starter CSV shipped with the internship repo — 30,000 pseudonymized pages across 32 clients, one row per page, built from 90-day GSC/GA4 aggregates (not the full ~79M-row warehouse; see the lane-pivot note in Section 3 for why). No client names, domains, or raw URLs appear anywhere in this file or in this notebook.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])

num_features = ["content_age_days", "days_since_last_update", "log_impressions_90d",
                 "avg_position", "ctr", "engagement_rate", "search_volume", "competition",
                 "word_count", "has_keyword_data", "has_word_count"]
cat_features = ["content_type", "main_intent"]

X_num = df[num_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_cat = pd.get_dummies(df[cat_features].fillna("unknown"), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1)
y = df["is_declining_label"]
groups = df["client_id"]

print("Label: is_declining_label = (trend_direction == 'down')")
print("Base rate:", round(y.mean(), 3))
print("Final feature set (", len(num_features + cat_features), "columns before dummy-encoding ):")
print(" ", num_features + cat_features)

Label: is_declining_label = (trend_direction == 'down')
Base rate: 0.542
Final feature set ( 13 columns before dummy-encoding ):
  ['content_age_days', 'days_since_last_update', 'log_impressions_90d', 'avg_position', 'ctr', 'engagement_rate', 'search_volume', 'competition', 'word_count', 'has_keyword_data', 'has_word_count', 'content_type', 'main_intent']


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

# Baseline: transparent rule from w04 -- stale (>=90 days) AND visible (300-30,000 impressions)
stale = df["days_since_last_update"] >= 90
visible = (df["impressions_90d"] >= 300) & (df["impressions_90d"] < 30000)
baseline_score_all = (stale & visible).astype(int) * df["impressions_90d"]

# Validation design: client-grouped 80/20 split -- no client's rows appear on both sides
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
print("Train clients:", groups.iloc[train_idx].nunique(), "| Test (held-out) clients:", groups.iloc[test_idx].nunique())
print("Client overlap between train/test:", len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])), "(must be 0)")

# Leakage confession test (w03/w06) -- re-confirmed here on the final feature set
def make_X(extra_cols):
    num = num_features + extra_cols
    Xn = df[num].replace([np.inf, -np.inf], np.nan).fillna(0)
    Xc = pd.get_dummies(df[cat_features].fillna("unknown"), drop_first=True)
    return pd.concat([Xn, Xc], axis=1)

print()
print("Leakage confession test (Logistic Regression AUC, grouped split):")
for label, extra in [("clean feature set", []), ("+ raw 30-day windows", ["impressions_last_30d", "impressions_prev_30d"]),
                      ("+ trend_pct itself", ["trend_pct"])]:
    Xc = make_X(extra)
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xc.iloc[train_idx]); Xte_s = scaler.transform(Xc.iloc[test_idx])
    m = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    m.fit(Xtr_s, y.iloc[train_idx])
    print(f"  {label}: AUC = {roc_auc_score(y.iloc[test_idx], m.predict_proba(Xte_s)[:,1]):.3f}")

Train clients: 25 | Test (held-out) clients: 7
Client overlap between train/test: 0 (must be 0)

Leakage confession test (Logistic Regression AUC, grouped split):
  clean feature set: AUC = 0.541
  + raw 30-day windows: AUC = 0.854
  + trend_pct itself: AUC = 1.000


**Assumptions:** a page's decline is visible in its own 90-day content-health signals (position, CTR, engagement, staleness) without needing causal attribution to any single change. **Label:** `is_declining_label = (trend_direction == 'down')`, FlyRank's own trend call, not something built from scratch. **Baseline:** the w04 transparent rule (stale AND visible), scored the same way the model is. **Split:** client-grouped 80/20, because clients are the unit that repeats — a random row split lets a client's other pages leak the answer (measured directly in Section 4). **Leakage:** trend_direction/trend_pct and their raw 30-day components are confirmed excluded above; no product-computed flags exist in this file to leak in the first place.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
from sklearn.model_selection import train_test_split

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Fit on the honest (grouped) training clients only
scaler = StandardScaler()
Xtr_s, Xte_s = scaler.fit_transform(X.iloc[train_idx]), scaler.transform(X.iloc[test_idx])
logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
logreg.fit(Xtr_s, y.iloc[train_idx])
rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                             class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

yte = y.iloc[test_idx]
results = {
    "base rate (majority class)": (None, yte.mean(), None),
    "baseline rule (w04)": (roc_auc_score(yte, baseline_score_all.iloc[test_idx]),
                             precision_at_k(baseline_score_all.iloc[test_idx], yte, 20),
                             precision_at_k(baseline_score_all.iloc[test_idx], yte, 50)),
    "logistic regression": (roc_auc_score(yte, logreg.predict_proba(Xte_s)[:,1]),
                             precision_at_k(logreg.predict_proba(Xte_s)[:,1], yte, 20),
                             precision_at_k(logreg.predict_proba(Xte_s)[:,1], yte, 50)),
    "random forest": (roc_auc_score(yte, rf.predict_proba(X.iloc[test_idx])[:,1]),
                       precision_at_k(rf.predict_proba(X.iloc[test_idx])[:,1], yte, 20),
                       precision_at_k(rf.predict_proba(X.iloc[test_idx])[:,1], yte, 50)),
}
print(f"{'method':24s} {'AUC':>6s} {'P@20':>6s} {'P@50':>6s}")
for name, (auc, p20, p50) in results.items():
    auc_s = f"{auc:.3f}" if auc is not None else "  --  "
    p20_s = f"{p20:.2f}" if not isinstance(p20, float) or True else ""
    print(f"{name:24s} {auc_s:>6s} {p20:6.2f} {p50:6.2f}")

print()
print("For contrast -- the same Random Forest, scored under a RANDOM row-level split instead:")
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
rf_leaky = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                                   class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf_leaky.fit(Xtr_r, ytr_r)
print("  Random-split AUC:", round(roc_auc_score(yte_r, rf_leaky.predict_proba(Xte_r)[:,1]), 3),
      " vs honest grouped-split AUC:", round(roc_auc_score(yte, rf.predict_proba(X.iloc[test_idx])[:,1]), 3))

method                      AUC   P@20   P@50
base rate (majority class)    --   0.51   0.51
baseline rule (w04)        0.492   0.45   0.38
logistic regression        0.541   0.50   0.56
random forest              0.591   0.50   0.56

For contrast -- the same Random Forest, scored under a RANDOM row-level split instead:
  Random-split AUC: 0.723  vs honest grouped-split AUC: 0.591


In [ ]:
from PIL import Image
for f in ["split_gap.png", "precision_at_50.png"]:
    img = Image.open(f"work/figures/{f}")
    print(f, "-", img.size)

split_gap.png - (900, 600)
precision_at_50.png - (900, 600)


**Reading the table.** On the 7 clients held out entirely from training, the Random Forest beats both the base rate and the transparent rule at precision@50 (0.56 vs. 0.51 vs. 0.38) and AUC (0.591 vs. 0.492 for the rule). The margin over the *base rate itself* is real but modest — this is a decision-support edge, not a dramatic one. The random-split contrast (0.723) shows why the split matters more than which model you pick: **31 of 32 clients** leaked across train/test under a naive random split, and that leak alone accounts for most of the apparent "skill" gain.

## 5. Limitations

*What this work cannot claim.*

In [ ]:
unseen_share = round((~df.index.isin(train_idx)).mean(), 3)
print("Share of the full 30,000-row portfolio from held-out (never-trained-on) clients:", unseen_share)
print("The precision@50 = 0.56 headline number is validated ONLY on that slice.")

Share of the full 30,000-row portfolio from held-out (never-trained-on) clients: 0.205


- **No causal claim.** This model ranks correlational risk; it does not show that refreshing a page *causes* recovery, and it was never designed to (would need a controlled before/after design per `DATA_USE.md`).
- **Small holdout.** The precision@50 = 0.56 headline rests on 7 held-out clients — a real, honest number, but not a large enough sample to trust its third decimal.
- **79% of the portfolio was seen in training.** Only the 20.5% of rows from held-out clients carry independent validation; scores for the other 79% would read optimistically if re-evaluated the same way.
- **One dominant client can swamp a pooled queue.** The action playbook (w07) found the top 50+ ranked rows belonged almost entirely to one genuinely high-decline client — real, but a human needs to catch that before assigning a whole review cycle to one account.
- **No signal for brand-new clients.** Every feature needs ~90 days of GSC/GA4 history; a client without that history isn't covered by anything here.
- **Not deployed against Google's actual ranking algorithm** — this model never claims to predict or explain algorithm behavior, only observed traffic/engagement trend.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
# Score the FULL portfolio with the honestly-trained model (fit on the 25 training clients only),
# then apply w07's reason-code logic.
df["rf_score"] = rf.predict_proba(X)[:, 1]
df["seen_in_training"] = df.index.isin(train_idx)

p50v, p90v = df["rf_score"].quantile(0.50), df["rf_score"].quantile(0.90)
rule_flag = stale & visible
df["risk_tier"] = pd.cut(df["rf_score"], bins=[-1, p50v, p90v, 2], labels=["low", "moderate", "high"])

df["reason_code"] = "low_priority"; df["action"] = "no_action"
both_high = rule_flag & (df["risk_tier"] == "high")
model_only = (~rule_flag) & (df["risk_tier"] == "high")
rule_mod = rule_flag & (df["risk_tier"] == "moderate")
rule_low_model = rule_flag & (df["risk_tier"] == "low")
df.loc[both_high, ["reason_code", "action"]] = ["stale_visible_and_top_risk_decile", "review_for_refresh_priority"]
df.loc[model_only, ["reason_code", "action"]] = ["top_risk_decile_not_flagged_by_rule", "review_for_refresh_secondary"]
df.loc[rule_mod, ["reason_code", "action"]] = ["stale_visible_moderate_model_risk", "review_for_refresh_secondary"]
df.loc[rule_low_model, ["reason_code", "action"]] = ["stale_visible_but_low_model_risk", "spot_check_only"]

print(df["action"].value_counts())
print()
print("Top priority: review_for_refresh_priority --", int((df['action']=='review_for_refresh_priority').sum()), "of 30,000 pages")

action
no_action                       21753
review_for_refresh_secondary     4081
spot_check_only                  2668
review_for_refresh_priority      1498
Name: count, dtype: int64

Top priority: review_for_refresh_priority -- 1498 of 30,000 pages


**What an editor does tomorrow:** start with the 1,498 `review_for_refresh_priority` pages (stale, visible, AND in the model's top risk decile), but **check for single-client concentration at the top first** — one very-stale client can otherwise fill an entire day's queue (Section 5). The 1,502 `top_risk_decile_not_flagged_by_rule` pages are the model's distinct catch beyond the rule — worth a second look, not equal-confidence to the first group. The 2,668 `spot_check_only` pages are where the rule and model disagree; never silently drop them, never fully automate them either.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
os.makedirs("work/figures", exist_ok=True)
os.makedirs("work/outputs", exist_ok=True)

queue_cols = ["content_id", "client_id", "rf_score", "risk_tier", "reason_code", "action",
              "seen_in_training", "days_since_last_update", "impressions_90d", "avg_position", "ctr"]
queue = df[queue_cols].sort_values("rf_score", ascending=False).reset_index(drop=True)
queue.to_csv("work/outputs/content_action_queue.csv", index=False)

results_table = pd.DataFrame([
    {"method": "base rate", "AUC": None, "P@20": round(yte.mean(),2), "P@50": round(yte.mean(),2)},
    {"method": "baseline rule (w04)", "AUC": 0.492, "P@20": 0.45, "P@50": 0.38},
    {"method": "logistic regression", "AUC": 0.541, "P@20": 0.50, "P@50": 0.56},
    {"method": "random forest", "AUC": 0.591, "P@20": 0.50, "P@50": 0.56},
])
results_table.to_csv("work/outputs/results_table.csv", index=False)

print("Figures already saved for the paper page:")
for f in os.listdir("work/figures"):
    print(" -", f"work/figures/{f}")
print()
print("Tables saved:")
print(" - work/outputs/content_action_queue.csv (", len(queue), "rows )")
print(" - work/outputs/results_table.csv")

Figures already saved for the paper page:
 - split_gap.png
 - precision_at_50.png
 - reason_code_breakdown.png

Tables saved:
 - work/outputs/content_action_queue.csv ( 30000 rows )
 - work/outputs/results_table.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.